In [ ]:
#| hide

from fh_matui.foundations import normalize_tokens, dedupe_preserve_order, stringify, listify, VEnum
from fh_matui.core import *
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
from fasthtml.common import A, Button as FhButton, I, Span


In [ ]:
#| hide

print("=== BeerCssChain Tests ===\n")

# Test 1: Basic initialization and string conversion
empty_chain = BeerCssChain()
assert str(empty_chain) == "", "Empty chain should produce empty string"
assert list(empty_chain) == [], "Empty chain should have no tokens"
print("✓ Empty chain initialization")

initial_chain = BeerCssChain(['primary', 'large'])
assert str(initial_chain) == "primary large", "Initial chain should join tokens with spaces"
assert list(initial_chain) == ['primary', 'large'], "Initial chain should preserve tokens"
print("✓ Chain initialization with tokens")

# Test 2: Single helper chaining
primary_chain = BeerCssChain().primary
assert str(primary_chain) == "primary", "Single helper should work"
print("✓ Single helper chaining")

large_chain = BeerCssChain().large
assert str(large_chain) == "large", "Large helper should work"

border_chain = BeerCssChain().border
assert str(border_chain) == "border", "Border helper should work"

# Test 3: Multiple helper chaining
multi_chain = BeerCssChain().primary.large.border
assert str(multi_chain) == "primary large border", "Multiple chaining should join all tokens"
print("✓ Multiple helper chaining")

complex_chain = BeerCssChain().secondary.medium.round.margin.elevate
chain_str = str(complex_chain)
assert 'secondary' in chain_str and 'medium' in chain_str and 'round' in chain_str, "Complex chain should contain all helpers"
print("✓ Complex chaining")

# Test 4: Underscore to dash conversion
underscore_chain = BeerCssChain().primary_text.large_margin.center_align
chain_str = str(underscore_chain)
assert 'primary-text' in chain_str or 'primary_text' in chain_str, "Should handle underscores"
assert 'large-margin' in chain_str or 'large_margin' in chain_str, "Should handle underscores"
assert 'center-align' in chain_str or 'center_align' in chain_str, "Should handle underscores"
print("✓ Underscore to dash conversion")

# Test 5: Chain continuation (immutability)
base_button = BeerCssChain().primary.large
enhanced_button = base_button.round.shadow
base_str = str(base_button)
enhanced_str = str(enhanced_button)
assert 'primary' in base_str and 'large' in base_str, "Base chain should remain unchanged"
assert 'round' in enhanced_str and 'shadow' in enhanced_str, "Enhanced chain should have new tokens"
print("✓ Chain continuation")

# Test 6: Iterator functionality
test_chain = BeerCssChain().primary.large.border.margin
tokens = list(test_chain)
assert len(tokens) == 4, "Should have 4 tokens"
assert 'primary' in tokens and 'large' in tokens, "Should contain expected tokens"
print("✓ Iterator functionality")

# Test 7: String representation
button_chain = BeerCssChain().primary.large.round
chain_str = str(button_chain)
assert isinstance(chain_str, str), "Should convert to string"
assert len(chain_str) > 0, "Should not be empty"
print("✓ String representation")

print("\n✅ All BeerCssChain tests passed!\n")

=== BeerCssChain Tests ===

✓ Empty chain initialization
✓ Chain initialization with tokens
✓ Single helper chaining
✓ Multiple helper chaining
✓ Complex chaining
✓ Underscore to dash conversion
✓ Chain continuation
✓ Iterator functionality
✓ String representation

✅ All BeerCssChain tests passed!



In [ ]:
#| hide

print("=== Theme Tests ===\n")

# Test 1: Theme color variants exist
assert hasattr(Theme, 'blue'), "Theme should have blue variant"
assert hasattr(Theme, 'red'), "Theme should have red variant"
assert hasattr(Theme, 'green'), "Theme should have green variant"
assert hasattr(Theme, 'amber'), "Theme should have amber variant"
print("✓ Theme color variants exist")

# Test 2: Theme.headers() returns list/tuple
blue_headers = Theme.blue.headers("Test App")
assert isinstance(blue_headers, (list, tuple)), "headers() should return list or tuple"
assert len(blue_headers) > 0, "headers() should not be empty"
print("✓ Theme.headers() returns collection")

# Test 3: Theme with title parameter
titled_headers = Theme.blue.headers(title="My App")
headers_str = str(titled_headers)
assert 'My App' in headers_str, "Title should be included in headers"
print("✓ Theme title parameter")

# Test 4: Theme with mode parameter
dark_headers = Theme.blue.headers(title="Dark App", mode="dark")
dark_str = str(dark_headers)
assert 'dark' in dark_str.lower(), "Mode should be included in headers"
print("✓ Theme mode parameter")

light_headers = Theme.blue.headers(title="Light App", mode="light")
light_str = str(light_headers)
assert 'light' in light_str.lower(), "Light mode should be included"
print("✓ Theme light mode")

# Test 5: Different theme colors produce different output
blue_str = str(Theme.blue.headers("App"))
red_str = str(Theme.red.headers("App"))
assert blue_str != red_str, "Different theme colors should produce different headers"
print("✓ Theme color variants produce different output")

# Test 6: Headers contain essential Beer CSS elements
headers = Theme.blue.headers("Test")
headers_str = str(headers)
# Should contain either meta tags or script tags for theme/mode
assert 'meta' in headers_str.lower() or 'script' in headers_str.lower(), "Headers should contain theme configuration"
print("✓ Headers contain theme configuration")

print("\n✅ All Theme tests passed!\n")

=== Theme Tests ===

✓ Theme color variants exist
✓ Theme.headers() returns collection
✓ Theme title parameter
✓ Theme mode parameter
✓ Theme light mode
✓ Theme color variants produce different output
✓ Headers contain theme configuration

✅ All Theme tests passed!



In [ ]:
#| hide
#| eval: false

import socket
import time
import subprocess

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        # Find process using the port
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        
        if result.stdout:
            # Extract PID from netstat output
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"✓ Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"⚠ Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=8888, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find an available port in range {start_port}-{start_port+max_attempts}")

# Stop existing server if running
if 'server' in globals(): 
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

# Try to kill any process on preferred port, then find available port
preferred_port = 8888
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app, rt = fast_app(hdrs=Theme.blue.headers("Blue App", mode="dark"))

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"✓ fast_app created with BLUE theme and dark mode on port {port}")
except Exception as e:
    print(f"✗ Failed to start server: {e}")
    raise

✓ fast_app created with BLUE theme and dark mode on port 8888


In [ ]:
#| hide
#| eval: false

preview(
    Div(
        H1("Theme Color Inheritance Demo", cls="primary"),
        P("Blue theme set in headers - see how components inherit colors"),
        
        # Mode switching section
        H2("Mode Controls"),
        Div(
            Button("Light Mode", onclick='setMode("light"); return false;', cls="border"),
            Button("Dark Mode", onclick='setMode("dark"); return false;', cls="border"),
            Button("Auto Mode", onclick='setMode("auto"); return false;', cls="border"),
            Button("Toggle Mode", onclick='toggleMode(); return false;', cls="border primary"),
            cls="margin"
        ),
     
    
        # Theme switching buttons
        H2("Theme Switching (see background + primary color changes)"),
        Div(
            Button("Blue Theme", onclick='setTheme("#2196f3"); return false;', cls="primary"),
            Button("Red Theme", onclick='setTheme("#f44336"); return false;', cls="border"),
            Button("Green Theme", onclick='setTheme("#4caf50"); return false;', cls="border"),
            Button("Amber Theme", onclick='setTheme("#ffc107"); return false;', cls="border"),
            cls="margin"
        ),
        
      
    )
)

In [ ]:
#| hide
#| eval: false

preview(
    Div(
        # Navigation sidebar
        Nav(
            Div(
                Button(I('menu'), onclick="toggleNav('#test-nav'); return false;", cls="circle transparent"),
                cls="padding"
            ),
            A(I('home'), Span('Home')),
            A(I('dashboard'), Span('Dashboard')),
            A(I('settings'), Span('Settings')),
            A(I('help'), Span('Help')),
            A(I('info'), Span('About')),
            cls="left l",
            id="test-nav"
        ),
        
        # Main content area
        Main(
            H2("Navigation Toggle Test (toggleNav function)"),
            H3("Content Area"),
            P("Click the menu button in the navigation to toggle it."),
            P("This demonstrates the custom toggleNav function working properly with Beer CSS navigation rails."),
            cls="responsive"
        )
    )
)